We'll **use DistilBERT** transformer, because it's smaller than BERT but the difference in the result will not be that big.

# Step 1: Data Prep

Decision 1: we'll use **raw text (not `final_message`)** because transformers can use punctuation, word order, capitalization.

Decision 2: for a correct comparison we'll use **the same training set** as baseline.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# just the raw message strings and the label
df = pd.read_csv('../data/SMSSpamCollection', sep='\t', header=None, names=['label', 'message'])
df = df.drop_duplicates()

y = df['label'].map({'ham': 0, 'spam': 1})

train_texts, test_texts, y_train, y_test = train_test_split(
    df['message'], y, test_size=0.2, random_state=42, stratify=y
)

# Step 2: Tokenization

DistillBURT uses **WordPiece tokenization**. It split strings into **subword units**, not just by spaces like in the baseline. Common whole words stay as a token, rare words are broken into tokens.

It's important because our dataset have a lot of messy real-world text message. Word-count-based methods (our baseline) do not handle well with this.

In [2]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# quick look at how it tokenizes one message
sample = train_texts.iloc[0]
tokens = tokenizer.tokenize(sample)
print(sample)
print(tokens)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Ta-Daaaaa! I am home babe, are you still up ?
['ta', '-', 'da', '##aa', '##aa', '!', 'i', 'am', 'home', 'babe', ',', 'are', 'you', 'still', 'up', '?']


We used most-used `uncased` for this project (and losing caps signal unlike baseline model).

# Step 3: Padding and Truncation

Batched matices requires every row to be the same length.

* **Padding**: special `[PAD]` token adding at the end of short message until they reach a target length'
* **Truncation**: messages longer than a target length get cut off.

To save computing resources, it is used **attention mask**: a second sequence of 1s (for real tokens) and 0s (for padding).

## Chosing the target length

In [3]:
# check how long messages actually are, in tokens, before picking a fixed length
token_lengths = train_texts.apply(lambda x: len(tokenizer.tokenize(x)))
print(token_lengths.describe())

count    4135.000000
mean       23.000967
std        16.949354
min         1.000000
25%        11.000000
50%        17.000000
75%        33.000000
max       219.000000
Name: message, dtype: float64


In [4]:
token_lengths.quantile(0.95)

np.float64(52.0)